# Task 05 — Deep Learning / NLP Text Classifier
## Movie Review Sentiment Classification using TensorFlow/Keras

This notebook satisfies the RABTECH Task 05 requirements: text vectorization, a multi-layer neural network with Dropout and Batch Normalization, Early Stopping, training/validation curves, test evaluation, and confidence-based predictions on unseen text.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

## 2. Load the IMDB Dataset
Label 0 = Negative and 1 = Positive.

In [ ]:
MAX_WORDS = 20000
(x_train_raw, y_train), (x_test_raw, y_test) = keras.datasets.imdb.load_data(num_words=MAX_WORDS)

print("Training samples:", len(x_train_raw))
print("Test samples:", len(x_test_raw))
print("First encoded review:", x_train_raw[0][:20])

## 3. Decode Integer Sequences into Text

In [ ]:
word_index = keras.datasets.imdb.get_word_index()
reverse_word_index = {value + 3: key for key, value in word_index.items()}
reverse_word_index[0] = "<PAD>"
reverse_word_index[1] = "<START>"
reverse_word_index[2] = "<UNK>"
reverse_word_index[3] = "<UNUSED>"

def decode_review(encoded_review):
    return " ".join(reverse_word_index.get(i, "<UNK>") for i in encoded_review)

print(decode_review(x_train_raw[0])[:1000])
print("Label:", "Positive" if y_train[0] == 1 else "Negative")

## 4. Create Training and Validation Sets
The official test set remains separate for final evaluation.

In [ ]:
rng = np.random.default_rng(42)
indices = rng.permutation(len(x_train_raw))
split = int(0.8 * len(indices))
train_idx = indices[:split]
val_idx = indices[split:]

x_train = [decode_review(x_train_raw[i]) for i in train_idx]
x_val = [decode_review(x_train_raw[i]) for i in val_idx]
x_test = [decode_review(x) for x in x_test_raw]

y_train_final = np.array(y_train)[train_idx]
y_val = np.array(y_train)[val_idx]
y_test_final = np.array(y_test)

print("Train:", len(x_train))
print("Validation:", len(x_val))
print("Test:", len(x_test))

## 5. Tokenize and Vectorize Text
TextVectorization converts raw text into fixed-length integer token sequences. It is adapted only on training text.

In [ ]:
MAX_TOKENS = 20000
SEQUENCE_LENGTH = 250

vectorizer = layers.TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode="int",
    output_sequence_length=SEQUENCE_LENGTH,
    standardize="lower_and_strip_punctuation"
)

vectorizer.adapt(tf.data.Dataset.from_tensor_slices(x_train).batch(256))
print("Vocabulary size:", len(vectorizer.get_vocabulary()))

## 6. Build a Multi-Layer Neural Network
The model uses Embedding, Dense layers, Dropout and Batch Normalization.

In [ ]:
model = keras.Sequential([
    vectorizer,
    layers.Embedding(MAX_TOKENS, 64),
    layers.GlobalAveragePooling1D(),
    layers.Dense(128, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.40),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.30),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 7. Train with Early Stopping

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    np.array(x_train), y_train_final,
    validation_data=(np.array(x_val), y_val),
    epochs=8,
    batch_size=256,
    callbacks=[early_stop],
    verbose=1
)

## 8. Training and Validation Accuracy Curve

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

## 9. Training and Validation Loss Curve

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

## 10. Evaluate on Unseen Test Data

In [ ]:
test_loss, test_accuracy = model.evaluate(np.array(x_test), y_test_final, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

## 11. Test Unseen Sample Inputs and Classification Confidence

In [ ]:
sample_reviews = [
    "This movie was fantastic. The story was engaging and the acting was excellent.",
    "I did not enjoy this movie. It was boring and the story was disappointing.",
    "A very entertaining film with great performances and a satisfying ending.",
    "The movie was slow, confusing and not worth watching."
]

probabilities = model.predict(np.array(sample_reviews), verbose=0).ravel()
results = []

for review, prob in zip(sample_reviews, probabilities):
    sentiment = "Positive" if prob >= 0.5 else "Negative"
    confidence = prob if prob >= 0.5 else 1 - prob
    results.append({
        "Review": review,
        "Predicted Sentiment": sentiment,
        "Confidence": round(float(confidence), 4)
    })

pd.DataFrame(results)

## 12. Reusable Prediction Function

In [ ]:
def predict_sentiment(text):
    probability = float(model.predict(np.array([text]), verbose=0)[0][0])
    sentiment = "Positive" if probability >= 0.5 else "Negative"
    confidence = probability if probability >= 0.5 else 1 - probability
    return sentiment, confidence

new_review = "The acting was wonderful and I really enjoyed the movie."
sentiment, confidence = predict_sentiment(new_review)

print("Review:", new_review)
print("Prediction:", sentiment)
print(f"Confidence: {confidence:.2%}")

## 13. Save the Trained Model

In [ ]:
model.save("task05_imdb_sentiment_classifier.keras")
print("Model saved as task05_imdb_sentiment_classifier.keras")

## Conclusion
A deep-learning NLP sentiment classifier was developed using TensorFlow/Keras. The pipeline includes text vectorization, word embeddings, a multi-layer neural network, Dropout, Batch Normalization, Early Stopping, training and validation curves, final test evaluation, and confidence-based predictions for unseen movie reviews.